# [Point-like Source Detection (`photutils.detection`)](https://photutils.readthedocs.io/en/latest/user_guide/detection.html)

One generally needs to identify astronomical sources in the data before performing photometry or other measurements. The [`photutils.detection`](https://photutils.readthedocs.io/en/latest/reference/detection_api.html#module-photutils.detection) subpackage provides tools to detect point-like (stellar) sources in an image. This subpackage also provides tools to find local peaks in an image that are above a specified threshold value.

For general-use source detection and extraction of both point-like and extended sources, please see [Image Segmentation](https://photutils.readthedocs.io/en/latest/user_guide/segmentation.html#image-segmentation).

## Detecting stars

Photutils includes two widely-used tools for detecting stars in an image, [DAOFIND](https://iraf.readthedocs.io/en/latest/tasks/noao/digiphot/apphot/daofind.html) and IRAF’s [starfind](https://iraf.readthedocs.io/en/latest/tasks/images/imcoords/starfind.html), plus a third tool that allows input of a custom user-defined kernel.

[`IRAFStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.IRAFStarFinder.html#photutils.detection.IRAFStarFinder) is a class that implements IRAF’s [starfind](https://iraf.readthedocs.io/en/latest/tasks/images/imcoords/starfind.html) algorithm. It is fundamentally similar to [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder), but [`IRAFStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.IRAFStarFinder.html#photutils.detection.IRAFStarFinder) always uses a circular Gaussian kernel whereas [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder) can use an elliptical Gaussian kernel. Another difference is that [`IRAFStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.IRAFStarFinder.html#photutils.detection.IRAFStarFinder) calculates the objects’ centroid, roundness, and sharpness using image moments.

The usage of [`IRAFStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.IRAFStarFinder.html#photutils.detection.IRAFStarFinder) and [`StarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.StarFinder.html#photutils.detection.StarFinder) follows the same pattern as [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder) shown below. Replace the class name and adjust the parameters (e.g., `fwhm and kernel`) as needed. Note that the `scale_threshold` parameter is specific to [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder). Note also that each class returns different output columns. For example, [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder) includes `daofind_mag` and `sharpness` columns, while [`IRAFStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.IRAFStarFinder.html#photutils.detection.IRAFStarFinder) includes `fwhm` and `pa` (position angle) columns. See each class’s API documentation for the full list of output columns.

As an example, let’s load a simulated HST star image and add Gaussian noise. We will then estimate the background and background noise using sigma-clipped statistics:

In [13]:
import numpy as np
from astropy.stats import sigma_clipped_stats
from photutils.datasets import load_simulated_hst_star_image, make_noise_image
from photutils.detection import DAOStarFinder

In [2]:
hdu = load_simulated_hst_star_image()
data = hdu.data + make_noise_image(hdu.data.shape, distribution='gaussian',
                                  mean=10.0, stddev=5, seed=0)
mean, median, std = sigma_clipped_stats(data, sigma=3.0)

print(np.array((mean, median, std)))

[10.44410657 10.39699777  5.09141794]


Now we will subtract the background and use an instance of [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder) to find the stars in the image that have FWHMs of around 2.5 pixels and have peaks approximately 5 times the background standard deviation above the background (i.e., the threshold is `5 * std`). The stars in the image are undersampled, so we will slightly relax the `sharpness_range` to allow for a wider range of values.

Running this class on the data yields an astropy [`QTable`](https://docs.astropy.org/en/stable/api/astropy.table.QTable.html#astropy.table.QTable) containing the results of the star finder:

In [14]:
threshold = 5.0 * std
daofind = DAOStarFinder(threshold, fwhm=2.5, sharpness_range=(0.2, 1.5))

TypeError: DAOStarFinder.__init__() got an unexpected keyword argument 'sharpness_range'

By default, [`DAOStarFinder`](https://photutils.readthedocs.io/en/latest/api/photutils.detection.DAOStarFinder.html#photutils.detection.DAOStarFinder) internally scales the input threshold by a factor derived from the convolution kernel to match the original [DAOFIND](https://iraf.readthedocs.io/en/latest/tasks/noao/digiphot/apphot/daofind.html) algorithm. To apply the threshold exactly as given (e.g., when supplying a spatial background-RMS map), set `scale_threshold=False`:

In [15]:
daofind_unscaled = DAOStarFinder(threshold, fwhm=2.5, 
                                sharpness_range=(0.2, 1.5),
                                scale_threshold=False)

TypeError: DAOStarFinder.__init__() got an unexpected keyword argument 'sharpness_range'

Running the finder on the background-subtracted data:

In [16]:
sources = daofind(data - median)
for col in sources.colnames:
    if col not in ('id', 'n_pixels'):
        sources[col].info.format = '%.2f'  # for consistent table output
sources.pprint(max_lines=12, max_width=76)

NameError: name 'daofind' is not defined

## API Reference

[Point-like Source Detection (photutils.detection)](https://photutils.readthedocs.io/en/latest/reference/detection_api.html)